# AIO2026 · M01 · Day 03 — Vector Database, RAG & Tìm ảnh tương đồng

Notebook gộp 2 bài thực hành Day 03 mình làm:

1. **Tìm ảnh tương đồng với OpenCV** — ảnh → vector → đo khoảng cách.
2. **Mô phỏng cơ chế RAG** — văn bản → embedding → truy xuất (cosine) → prompt.

> Điểm thú vị nhất: cùng 1 ảnh truy vấn, đo bằng **Euclidean (L2)** và **Cosine** lại cho 2 đáp án khác nhau — phần 2.3 sẽ mổ xẻ vì sao.

## 0. Cài đặt thư viện

Trên Google Colab, bỏ comment dòng `pip` để cài. Trên máy đã có sẵn thì bỏ qua.

In [ ]:
# !pip install numpy opencv-python sentence-transformers
import cv2
import numpy as np

print("numpy", np.__version__, "| opencv", cv2.__version__)

## 1. Các hàm đo khoảng cách / độ tương đồng

Mọi dữ liệu (ảnh, văn bản) cuối cùng đều thành **vector**. So sánh = đo khoảng cách giữa vector.

- **L1 (Manhattan):** tổng trị tuyệt đối các hiệu. Nhỏ → giống.
- **L2 (Euclidean):** độ dài đường thẳng giữa 2 vector. Nhỏ → giống.
- **Cosine Similarity:** cos góc giữa 2 vector (bỏ qua độ dài). Gần 1 → giống.

In [ ]:
def l1_distance(v1, v2):
    """Tổng |hiệu|. Càng nhỏ càng giống."""
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    if v1.shape != v2.shape:  # edge case: khác số chiều
        raise ValueError(f"Khác số chiều: {v1.shape} != {v2.shape}")
    return float(np.sum(np.abs(v1 - v2)))


def l2_distance(v1, v2):
    """Khoảng cách Euclid. Càng nhỏ càng giống."""
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    if v1.shape != v2.shape:
        raise ValueError(f"Khác số chiều: {v1.shape} != {v2.shape}")
    return float(np.linalg.norm(v1 - v2))


def cosine_similarity(v1, v2):
    """Cos góc giữa 2 vector, trong [-1, 1]. Vector 0 -> quy ước 0.0 (tránh chia 0)."""
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    if v1.shape != v2.shape:
        raise ValueError(f"Khác số chiều: {v1.shape} != {v2.shape}")
    na, nb = np.linalg.norm(v1), np.linalg.norm(v2)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(v1, v2) / (na * nb))

In [ ]:
# Kiểm tra nhanh
print("L1:", l1_distance([0, 0], [3, 4]))  # 7
print("L2:", l2_distance([0, 0], [3, 4]))  # 5 (tam giác 3-4-5)
print("cos cùng hướng :", cosine_similarity([1, 0], [2, 0]))  # 1.0
print("cos vuông góc  :", cosine_similarity([1, 0], [0, 9]))  # 0.0

## 2. Tìm ảnh tương đồng với OpenCV

Quy trình: đọc ảnh **xám** → resize **64×64** → duỗi phẳng thành vector **4096 chiều** → chuẩn hóa [0,1] → so khoảng cách.

### 2.1. Tiền xử lý 1 ảnh

In [ ]:
import os


def process_image(image_path, size=(64, 64)):
    """Ảnh -> vector đặc trưng đã chuẩn hóa [0,1]. Báo lỗi rõ nếu file thiếu/hỏng."""
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)  # đọc thẳng ảnh xám (1 kênh)
    if img is None:  # file tồn tại nhưng không decode được
        raise ValueError(f"Không giải mã được ảnh (hỏng?): {image_path}")
    img = cv2.resize(img, size)  # đưa mọi ảnh về cùng kích thước
    return img.flatten().astype(float) / 255.0  # duỗi phẳng + chuẩn hóa độ sáng

### 2.2. Quét cả thư mục, tìm ảnh giống nhất

Trên Colab: upload thư mục `Data` (gồm `query.jpg` và `images_folder/`) rồi sửa `DATA_DIR` cho đúng.

In [ ]:
DATA_DIR = "Data"  # sửa lại path nếu chạy ở Colab
SUPPORTED = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def search_similar(query_path, dataset_dir, size=(64, 64), metric="l2", top_k=3):
    """Trả về top_k (tên_file, điểm) giống nhất. Bỏ qua ảnh hỏng, không làm sập."""
    fn = {"l1": l1_distance, "l2": l2_distance, "cosine": cosine_similarity}[metric]
    higher_better = metric == "cosine"  # cosine: cao = giống; L1/L2: thấp = giống
    q = process_image(query_path, size)
    results = []
    for name in sorted(os.listdir(dataset_dir)):
        if not name.lower().endswith(SUPPORTED):
            continue
        try:
            v = process_image(os.path.join(dataset_dir, name), size)
        except (ValueError, FileNotFoundError) as e:
            print("[BỎ QUA]", name, "-", e)
            continue
        results.append((name, fn(q, v)))
    results.sort(key=lambda x: x[1], reverse=higher_better)
    return results[:top_k]


query = os.path.join(DATA_DIR, "query.jpg")
imgs = os.path.join(DATA_DIR, "images_folder")
print("Top 3 theo L2:")
for name, score in search_similar(query, imgs, metric="l2"):
    print(f"  {name:12s} {score:.4f}")

### 2.3. 🔍 Điểm hay: L2 vs Cosine cho đáp án KHÁC nhau

Chạy cả hai metric trên cùng ảnh truy vấn và so sánh ảnh đứng đầu.

In [ ]:
top_l2 = search_similar(query, imgs, metric="l2", top_k=1)[0]
top_cos = search_similar(query, imgs, metric="cosine", top_k=1)[0]
print("Giống nhất theo L2     :", top_l2)
print("Giống nhất theo Cosine :", top_cos)

**Vì sao khác?** Vector ảnh = độ sáng từng pixel.

- **L2 (Euclidean)** cộng dồn chênh lệch độ sáng *từng pixel* → một ảnh sáng/tối hơn dù **cùng hình dạng** vẫn bị đẩy khoảng cách lên cao.
- **Cosine** chỉ đo *hướng* của vector (xu hướng phân bố sáng–tối) → **bất biến với độ sáng tổng thể**.

Ví von: L2 hỏi *'hai điểm cách nhau bao xa?'*, Cosine hỏi *'hai mũi tên có cùng hướng không?'*. Tăng sáng cả ảnh = kéo dài mũi tên → L2 đổi, hướng không đổi.

➡️ Khi *cường độ tuyệt đối không quan trọng, chỉ cần đúng hình dạng/ngữ nghĩa* → **Cosine đáng tin hơn**. Đó cũng là lý do RAG (phần 3) dùng Cosine.

## 3. Mô phỏng cơ chế RAG (Retrieval-Augmented Generation)

Pipeline 3 bước: **Indexing** (văn bản → vector) → **Retrieval** (chọn ngữ cảnh gần nhất bằng cosine) → **Generation** (nhét ngữ cảnh vào prompt cho mô hình ngôn ngữ lớn).

### 3.1. Indexing — văn bản tiếng Việt thành vector
Dùng mô hình embedding tiếng Việt `keepitreal/vietnamese-sbert` qua thư viện `sentence-transformers`.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("keepitreal/vietnamese-sbert")

documents = [
    "Khóa học AIO2026 gồm có 10 module chuyên sâu về Trí tuệ Nhân tạo.",
    "Tài liệu hướng dẫn thủ tục đăng ký nhập học và đóng học phí trực tuyến.",
    "Hướng dẫn cấu trúc dữ liệu và giải thuật với Python để xử lý mảng.",
]
vector_db = model.encode(documents)
print("Kích thước vector DB:", vector_db.shape)  # (3, 768): 3 câu, mỗi câu 768 chiều

### 3.2. Retrieval — tìm ngữ cảnh liên quan nhất bằng Cosine Similarity

In [ ]:
def retrieve(query, top_k=1):
    """Trả về top_k (tài liệu, điểm cosine) liên quan nhất tới câu hỏi."""
    if not query or not query.strip():  # edge case: query rỗng
        raise ValueError("Câu truy vấn rỗng.")
    q = model.encode(query)
    # cosine vector hóa: (db·q) / (||db|| * ||q||) cho từng dòng
    scores = vector_db @ q / (np.linalg.norm(vector_db, axis=1) * np.linalg.norm(q))
    idx = np.argsort(scores)[::-1][: min(top_k, len(documents))]  # clamp top_k
    return [(documents[i], float(scores[i])) for i in idx]


query = "Khóa học AIO2026 có cấu trúc bao nhiêu phần?"
for doc, s in retrieve(query, top_k=3):
    print(f"{s:.4f}  {doc}")

### 3.3. Generation — nhét ngữ cảnh vào prompt

Ở bước này ta chỉ *dựng prompt*; prompt này mới được gửi cho mô hình ngôn ngữ lớn (ChatGPT/Gemini...) để sinh câu trả lời có căn cứ.

In [ ]:
context = retrieve(query, top_k=1)[0][0]
prompt = f"""Dựa vào ngữ cảnh được cung cấp dưới đây để trả lời câu hỏi.
Tuyệt đối không sử dụng thông tin bên ngoài. Nếu ngữ cảnh không đủ, trả lời 'Không đủ thông tin'.

Ngữ cảnh: {context}
Câu hỏi: {query}
Trả lời:"""
print(prompt)